### **Extracción de datos desde una tabla HTML importada en un Dataframe**

#### **`Enunciado 1`: Queremos obtener todos los tipos de datos que son "mutables"**

##### **Carga de tabla HTML en un Dataframe** 

Nos muestra en una `lista` de **TODAS LAS TABLAS QUE SE ENCUENTRAN EN LA URL** en distintos Dataframes:

In [1]:
import pandas
import re

url = "https://en.wikipedia.org/wiki/Python_(programming_language)"
df = pandas.read_html(url)
df

[                                                    0  \
 0                                                 NaN   
 1                                            Paradigm   
 2                                         Designed by   
 3                                           Developer   
 4                                      First appeared   
 5                                                 NaN   
 6                                      Stable release   
 7                                   Typing discipline   
 8                                                  OS   
 9                                             License   
 10                                Filename extensions   
 11                                            Website   
 12                              Major implementations   
 13  CPython, PyPy, Stackless Python, MicroPython, ...   
 14                                           Dialects   
 15                       Cython, RPython, Starlark[9]   
 16           

Vamos a utilizar el segundo elemento de la lista que corresponde a la siguiente tabla:

In [2]:
df_tabla = pandas.read_html(url)[1]
df_tabla

,Type,Mutability,Description,Syntax examples
0,bool,immutable,Boolean value,True False
1,bytearray,mutable,Sequence of bytes,"bytearray(b'Some ASCII') bytearray(b""Some ASCI..."
2,bytes,immutable,Sequence of bytes,"b'Some ASCII' b""Some ASCII"" bytes([119, 105, 1..."
3,complex,immutable,Complex number with real and imaginary parts,3+2.7j 3 + 2.7j
4,dict,mutable,Associative array (or dictionary) of key and v...,"{'key1': 1.0, 3: False} {}"
5,types.EllipsisType,immutable,An ellipsis placeholder to be used as an index...,... Ellipsis
6,float,immutable,Double-precision floating-point number. The pr...,1.33333
7,frozenset,immutable,"Unordered set, contains no duplicates; can con...","frozenset([4.0, 'string', True])"
8,int,immutable,Integer of unlimited magnitude[114],42
9,list,mutable,"List, can contain mixed types","[4.0, 'string', True] []"


##### **Carga del Dataframe como un string** 

In [8]:
from pprint import pprint
data = df_tabla.to_string()
print(type(data))
pprint(data)

<class 'str'>
('                        Type '
 'Mutability                                                                                                                                                                      '
 'Description                                                                    '
 'Syntax examples\n'
 '0                       bool  '
 'immutable                                                                                                                                                                    '
 'Boolean '
 'value                                                                         '
 'True False\n'
 '1                  bytearray    '
 'mutable                                                                                                                                                                '
 'Sequence of bytes  bytearray(b\'Some ASCII\') bytearray(b"Some ASCII") '
 'bytearray([119, 105, 107, 105])\n'
 '2                      byte

##### **Desarrollo**

Supongamos que la primera tarea de filtrado es obtener todos los tipos de datos. Esos serían los que están en la segunda columna de la tabla si te fijas bien llamada "Type". Así que queremos todos los tipos de datos que son "mutables" en Python. Para esto, necesitaríamos de alguna manera referenciar la segunda columna y crear nuestro filtro regex alrededor de los valores en esa columna. Así que básicamente las dos columnas que nos interesan principalmente son "Type" y "Mutability". Por otra parte, también tenemos que tener este dataframe como una cadena con el fin de trabajar adecuadamente con expresiones regulares.

Nos damos cuenta de que cada línea de la tabla, excepto la cabecera de la tabla, que no nos interesa realmente, cada línea comienza con \n, para la primera línea, por ejemplo, donde \n es el carácter de nueva línea. Por esta razón, he comenzado mi patrón con un carácter \n para señalar el comienzo de una nueva línea. Yo podría haber hecho esto de otras maneras, también, lo sé, pero yo quería que mi patrón para ser lo más explícito posible en este momento. A continuación, tenemos el "ID" en la primera columna de la tabla, por lo que sería un número entero, que puede ser un solo dígito o dos o más dígitos. Precisamente por eso he utilizado la secuencia especial \d seguida de un 1 y una coma entre llaves para señalar que el número de dígitos aquí es al menos uno. A continuación, después del "ID" y básicamente entre cada dos columnas consecutivas de la tabla, tenemos espacios en blanco. Aquí, podemos tener un solo espacio en blanco o múltiples espacios en blanco. Además, decidimos utilizar la secuencia especial \s, que cubre todos los caracteres de espacio en blanco, incluyendo el espacio regular del teclado o el carácter de tabulación (\t), por ejemplo, también seguido de un 1 y una coma entre llaves, porque podemos tener uno o más espacios en blanco en este punto del patrón de expresión regular. A continuación, después de la primera aparición de un espacio en blanco en cada fila, tenemos el tipo de datos de Python que nos interesa. Por eso he incluido el patrón estándar para el margen de texto, un punto, el signo más y un signo de interrogación (.+?) entre paréntesis para señalar un grupo. Como recordatorio, el punto (.) representa cualquier carácter excepto el carácter de nueva línea (\n). A continuación, el signo más (+) significa una o más ocurrencias del patrón anterior. Por tanto, sería uno o más caracteres. Y finalmente, el signo de interrogación, como probablemente ya sepa, controla el comportamiento codicioso para este sub-patrón. Ahora, después de nuestro grupo entre paréntesis, tenemos otro conjunto de uno o más espacios en blanco, de nuevo, representado por la secuencia especial \s seguida de un par de llaves. A continuación, dijimos que queríamos obtener todos los tipos de datos "mutables" de esta tabla. Así que, dado que la siguiente columna se refiere a la mutabilidad de cada tipo de datos, podemos especificar aquí la palabra exacta que nos interesa. Sí, podemos ser lo más específicos posible en este caso para mantener las cosas simples y fáciles de leer y entender. Por lo tanto, he utilizado la palabra "mutable" porque eso es lo que estamos buscando. Por último, en cada línea en el Dataframe, tenemos otra información también, que no es relevante para nosotros en este escenario. Así que por eso decidí referirme a ella genéricamente usando un punto y un signo más (.+). Por último, pero no menos importante, después del patrón y la coma, no se olvide de añadir la variable que hace referencia a nuestro string objetivo `data`.

In [4]:
s1 = re.findall(r"\n\d{1,}\s{1,}(.+?)\s{1,}mutable.+", data)
s1

['bytearray', 'dict', 'list', 'set']

#### **`Enunciado 2`: Queremos obtener todos los tipos de datos para aquellos donde el valor de la última columna "Syntax examples" contenga una llave de apertura y cierre {}**

##### **Carga de tabla HTML en un Dataframe** 

Nos muestra en una `lista` de **TODAS LAS TABLAS QUE SE ENCUENTRAN EN LA URL** en distintos Dataframes:

In [9]:
import pandas
import re

url = "https://en.wikipedia.org/wiki/Python_(programming_language)"
df = pandas.read_html(url)
df

[                                                    0  \
 0                                                 NaN   
 1                                            Paradigm   
 2                                         Designed by   
 3                                           Developer   
 4                                      First appeared   
 5                                                 NaN   
 6                                      Stable release   
 7                                   Typing discipline   
 8                                                  OS   
 9                                             License   
 10                                Filename extensions   
 11                                            Website   
 12                              Major implementations   
 13  CPython, PyPy, Stackless Python, MicroPython, ...   
 14                                           Dialects   
 15                       Cython, RPython, Starlark[9]   
 16           

Vamos a utilizar el segundo elemento de la lista que corresponde a la siguiente tabla:

In [10]:
df_tabla = pandas.read_html(url)[1]
df_tabla

,Type,Mutability,Description,Syntax examples
0,bool,immutable,Boolean value,True False
1,bytearray,mutable,Sequence of bytes,"bytearray(b'Some ASCII') bytearray(b""Some ASCI..."
2,bytes,immutable,Sequence of bytes,"b'Some ASCII' b""Some ASCII"" bytes([119, 105, 1..."
3,complex,immutable,Complex number with real and imaginary parts,3+2.7j 3 + 2.7j
4,dict,mutable,Associative array (or dictionary) of key and v...,"{'key1': 1.0, 3: False} {}"
5,types.EllipsisType,immutable,An ellipsis placeholder to be used as an index...,... Ellipsis
6,float,immutable,Double-precision floating-point number. The pr...,1.33333
7,frozenset,immutable,"Unordered set, contains no duplicates; can con...","frozenset([4.0, 'string', True])"
8,int,immutable,Integer of unlimited magnitude[114],42
9,list,mutable,"List, can contain mixed types","[4.0, 'string', True] []"


##### **Carga del Dataframe como un string** 

In [11]:
from pprint import pprint
data = df_tabla.to_string()
print(type(data))
pprint(data)

<class 'str'>
('                        Type '
 'Mutability                                                                                                                                                                      '
 'Description                                                                    '
 'Syntax examples\n'
 '0                       bool  '
 'immutable                                                                                                                                                                    '
 'Boolean '
 'value                                                                         '
 'True False\n'
 '1                  bytearray    '
 'mutable                                                                                                                                                                '
 'Sequence of bytes  bytearray(b\'Some ASCII\') bytearray(b"Some ASCII") '
 'bytearray([119, 105, 107, 105])\n'
 '2                      byte

##### **Test**

In [14]:
s2 = re.findall(r"\n\d{1,}\s{1,}(\w+?)\s{1,}.+\{.*\}", data)
s2

['dict', 'set']

##### **Desarrollo**

In [13]:
# No es necesario especificar el \n
s2 = re.findall(r"\d{1,}\s{1,}(\w+?)\s{1,}.+\{.*\}", data)
s2

['dict', 'set']

#### **`Enunciado 3`: Queremos encontrar y devolver la descripción correspondiente a cada tipo de datos cuyo nombre no supere los cuatro caracteres. Por tanto, serían tipos de datos como bool, list o set, entre otros**

##### **Carga de tabla HTML en un Dataframe** 

Nos muestra en una `lista` de **TODAS LAS TABLAS QUE SE ENCUENTRAN EN LA URL** en distintos Dataframes:

In [15]:
import pandas
import re

url = "https://en.wikipedia.org/wiki/Python_(programming_language)"
df = pandas.read_html(url)
df

[                                                    0  \
 0                                                 NaN   
 1                                            Paradigm   
 2                                         Designed by   
 3                                           Developer   
 4                                      First appeared   
 5                                                 NaN   
 6                                      Stable release   
 7                                   Typing discipline   
 8                                                  OS   
 9                                             License   
 10                                Filename extensions   
 11                                            Website   
 12                              Major implementations   
 13  CPython, PyPy, Stackless Python, MicroPython, ...   
 14                                           Dialects   
 15                       Cython, RPython, Starlark[9]   
 16           

Vamos a utilizar el segundo elemento de la lista que corresponde a la siguiente tabla:

In [16]:
df_tabla = pandas.read_html(url)[1]
df_tabla

,Type,Mutability,Description,Syntax examples
0,bool,immutable,Boolean value,True False
1,bytearray,mutable,Sequence of bytes,"bytearray(b'Some ASCII') bytearray(b""Some ASCI..."
2,bytes,immutable,Sequence of bytes,"b'Some ASCII' b""Some ASCII"" bytes([119, 105, 1..."
3,complex,immutable,Complex number with real and imaginary parts,3+2.7j 3 + 2.7j
4,dict,mutable,Associative array (or dictionary) of key and v...,"{'key1': 1.0, 3: False} {}"
5,types.EllipsisType,immutable,An ellipsis placeholder to be used as an index...,... Ellipsis
6,float,immutable,Double-precision floating-point number. The pr...,1.33333
7,frozenset,immutable,"Unordered set, contains no duplicates; can con...","frozenset([4.0, 'string', True])"
8,int,immutable,Integer of unlimited magnitude[114],42
9,list,mutable,"List, can contain mixed types","[4.0, 'string', True] []"


##### **Carga del Dataframe como un string** 

In [17]:
from pprint import pprint
data = df_tabla.to_string()
print(type(data))
pprint(data)

<class 'str'>
('                        Type '
 'Mutability                                                                                                                                                                      '
 'Description                                                                    '
 'Syntax examples\n'
 '0                       bool  '
 'immutable                                                                                                                                                                    '
 'Boolean '
 'value                                                                         '
 'True False\n'
 '1                  bytearray    '
 'mutable                                                                                                                                                                '
 'Sequence of bytes  bytearray(b\'Some ASCII\') bytearray(b"Some ASCII") '
 'bytearray([119, 105, 107, 105])\n'
 '2                      byte

##### **Test**

In [21]:
s3 = re.findall(r"\n\d{1,}\s{1,}\w{1,4}\s{1,}\w+?\s{1,}(.+?)\s{2,}", data)
s3

['Boolean value',
 'Associative array (or dictionary) of key and value pairs; can contain mixed types (keys and values), keys must be a hashable type',
 'Integer of unlimited magnitude[114]',
 'List, can contain mixed types',
 'Unordered set, contains no duplicates; can contain mixed types, if hashable',
 'A character string: sequence of Unicode codepoints']

##### **Desarrollo**

In [20]:
# No es necesario especificar el \n como inicio de linea
s3 = re.findall(r"\d{1,}\s{1,}\w{1,4}\s{1,}\w+?\s{1,}(.+?)\s{2,}", data)
s3

['Boolean value',
 'Associative array (or dictionary) of key and value pairs; can contain mixed types (keys and values), keys must be a hashable type',
 'precision.[113]',
 'immutable',
 'Integer of unlimited magnitude[114]',
 'mutable',
 'Unordered set, contains no duplicates; can contain mixed types, if hashable',
 'A character string: sequence of Unicode codepoints']

#### **`Enunciado 4`: Queremos devolver los 10 primeros caracteres de la descripción para los tipos cuyos ID son números impares, números como 1, 3, 5, 7 y 9, también 11, 13 y 15 son números impares también. Así que no debemos omitirlos**

##### **Carga de tabla HTML en un Dataframe** 

Nos muestra en una `lista` de **TODAS LAS TABLAS QUE SE ENCUENTRAN EN LA URL** en distintos Dataframes:

In [22]:
import pandas
import re

url = "https://en.wikipedia.org/wiki/Python_(programming_language)"
df = pandas.read_html(url)
df

[                                                    0  \
 0                                                 NaN   
 1                                            Paradigm   
 2                                         Designed by   
 3                                           Developer   
 4                                      First appeared   
 5                                                 NaN   
 6                                      Stable release   
 7                                   Typing discipline   
 8                                                  OS   
 9                                             License   
 10                                Filename extensions   
 11                                            Website   
 12                              Major implementations   
 13  CPython, PyPy, Stackless Python, MicroPython, ...   
 14                                           Dialects   
 15                       Cython, RPython, Starlark[9]   
 16           

Vamos a utilizar el segundo elemento de la lista que corresponde a la siguiente tabla:

In [23]:
df_tabla = pandas.read_html(url)[1]
df_tabla

,Type,Mutability,Description,Syntax examples
0,bool,immutable,Boolean value,True False
1,bytearray,mutable,Sequence of bytes,"bytearray(b'Some ASCII') bytearray(b""Some ASCI..."
2,bytes,immutable,Sequence of bytes,"b'Some ASCII' b""Some ASCII"" bytes([119, 105, 1..."
3,complex,immutable,Complex number with real and imaginary parts,3+2.7j 3 + 2.7j
4,dict,mutable,Associative array (or dictionary) of key and v...,"{'key1': 1.0, 3: False} {}"
5,types.EllipsisType,immutable,An ellipsis placeholder to be used as an index...,... Ellipsis
6,float,immutable,Double-precision floating-point number. The pr...,1.33333
7,frozenset,immutable,"Unordered set, contains no duplicates; can con...","frozenset([4.0, 'string', True])"
8,int,immutable,Integer of unlimited magnitude[114],42
9,list,mutable,"List, can contain mixed types","[4.0, 'string', True] []"


##### **Carga del Dataframe como un string** 

In [24]:
from pprint import pprint
data = df_tabla.to_string()
print(type(data))
pprint(data)

<class 'str'>
('                        Type '
 'Mutability                                                                                                                                                                      '
 'Description                                                                    '
 'Syntax examples\n'
 '0                       bool  '
 'immutable                                                                                                                                                                    '
 'Boolean '
 'value                                                                         '
 'True False\n'
 '1                  bytearray    '
 'mutable                                                                                                                                                                '
 'Sequence of bytes  bytearray(b\'Some ASCII\') bytearray(b"Some ASCII") '
 'bytearray([119, 105, 107, 105])\n'
 '2                      byte

##### **Test**

In [80]:
string = "0\n1\n2\n3\n4\n5\n6\n7\n8\n9\n10\n11\n12"
s = re.findall(r"\d*[13579]", string)
print(s)

['1', '3', '5', '7', '9', '1', '11', '1']


In [27]:
s4 = re.findall(r"\n\d*[13579]\s{1,}\w+?\s{1,}\w+?\s{1,}(.{10})", data)
s4

['Sequence o',
 'Complex nu',
 'Unordered ',
 'List, can ',
 'Unordered ',
 'Can contai']

##### **Desarrollo**

In [28]:
# No es necesario especificar el \n como inicio de linea
s4 = re.findall(r"\d*[13579]\s{1,}\w+?\s{1,}\w+?\s{1,}(.{10})", data)
s4

['Sequence o',
 'Complex nu',
 'precision.',
 'immutable ',
 'List, can ',
 'Unordered ',
 'Can contai']

#### **`Enunciado 5`: Queremos devolver todos los tipos del dataframe para los que la columna "Syntax examples" contenga al menos un número flotante**

##### **Carga de tabla HTML en un Dataframe** 

Nos muestra en una `lista` de **TODAS LAS TABLAS QUE SE ENCUENTRAN EN LA URL** en distintos Dataframes:

In [29]:
import pandas
import re

url = "https://en.wikipedia.org/wiki/Python_(programming_language)"
df = pandas.read_html(url)
df

[                                                    0  \
 0                                                 NaN   
 1                                            Paradigm   
 2                                         Designed by   
 3                                           Developer   
 4                                      First appeared   
 5                                                 NaN   
 6                                      Stable release   
 7                                   Typing discipline   
 8                                                  OS   
 9                                             License   
 10                                Filename extensions   
 11                                            Website   
 12                              Major implementations   
 13  CPython, PyPy, Stackless Python, MicroPython, ...   
 14                                           Dialects   
 15                       Cython, RPython, Starlark[9]   
 16           

Vamos a utilizar el segundo elemento de la lista que corresponde a la siguiente tabla:

In [30]:
df_tabla = pandas.read_html(url)[1]
df_tabla

,Type,Mutability,Description,Syntax examples
0,bool,immutable,Boolean value,True False
1,bytearray,mutable,Sequence of bytes,"bytearray(b'Some ASCII') bytearray(b""Some ASCI..."
2,bytes,immutable,Sequence of bytes,"b'Some ASCII' b""Some ASCII"" bytes([119, 105, 1..."
3,complex,immutable,Complex number with real and imaginary parts,3+2.7j 3 + 2.7j
4,dict,mutable,Associative array (or dictionary) of key and v...,"{'key1': 1.0, 3: False} {}"
5,types.EllipsisType,immutable,An ellipsis placeholder to be used as an index...,... Ellipsis
6,float,immutable,Double-precision floating-point number. The pr...,1.33333
7,frozenset,immutable,"Unordered set, contains no duplicates; can con...","frozenset([4.0, 'string', True])"
8,int,immutable,Integer of unlimited magnitude[114],42
9,list,mutable,"List, can contain mixed types","[4.0, 'string', True] []"


##### **Carga del Dataframe como un string** 

In [31]:
from pprint import pprint
data = df_tabla.to_string()
print(type(data))
pprint(data)

<class 'str'>
('                        Type '
 'Mutability                                                                                                                                                                      '
 'Description                                                                    '
 'Syntax examples\n'
 '0                       bool  '
 'immutable                                                                                                                                                                    '
 'Boolean '
 'value                                                                         '
 'True False\n'
 '1                  bytearray    '
 'mutable                                                                                                                                                                '
 'Sequence of bytes  bytearray(b\'Some ASCII\') bytearray(b"Some ASCII") '
 'bytearray([119, 105, 107, 105])\n'
 '2                      byte

##### **Test**

In [37]:
s5 = re.findall(r"\d{1,}\s{1,}(\w+?)\s{1,}.+\d{1,}\.\d{1,}", data)
s5

['complex', 'dict', 'float', 'frozenset', '9', 'set', 'tuple']

##### **Desarrollo**

In [36]:
# Si es necesario especificar el \n como inicio de linea
s5 = re.findall(r"\n\d{1,}\s{1,}(\w+?)\s{1,}.+\d{1,}\.\d{1,}", data)
s5

['complex', 'dict', 'float', 'frozenset', 'list', 'set', 'tuple']

#### **`Enunciado 6`: Queremos obtener todos los tipos de datos que son "inmutables"**

##### **Carga de tabla HTML en un Dataframe** 

Nos muestra en una `lista` de **TODAS LAS TABLAS QUE SE ENCUENTRAN EN LA URL** en distintos Dataframes:

In [38]:
import pandas
import re

url = "https://en.wikipedia.org/wiki/Python_(programming_language)"
df = pandas.read_html(url)
df

[                                                    0  \
 0                                                 NaN   
 1                                            Paradigm   
 2                                         Designed by   
 3                                           Developer   
 4                                      First appeared   
 5                                                 NaN   
 6                                      Stable release   
 7                                   Typing discipline   
 8                                                  OS   
 9                                             License   
 10                                Filename extensions   
 11                                            Website   
 12                              Major implementations   
 13  CPython, PyPy, Stackless Python, MicroPython, ...   
 14                                           Dialects   
 15                       Cython, RPython, Starlark[9]   
 16           

Vamos a utilizar el segundo elemento de la lista que corresponde a la siguiente tabla:

In [39]:
df_tabla = pandas.read_html(url)[1]
df_tabla

,Type,Mutability,Description,Syntax examples
0,bool,immutable,Boolean value,True False
1,bytearray,mutable,Sequence of bytes,"bytearray(b'Some ASCII') bytearray(b""Some ASCI..."
2,bytes,immutable,Sequence of bytes,"b'Some ASCII' b""Some ASCII"" bytes([119, 105, 1..."
3,complex,immutable,Complex number with real and imaginary parts,3+2.7j 3 + 2.7j
4,dict,mutable,Associative array (or dictionary) of key and v...,"{'key1': 1.0, 3: False} {}"
5,types.EllipsisType,immutable,An ellipsis placeholder to be used as an index...,... Ellipsis
6,float,immutable,Double-precision floating-point number. The pr...,1.33333
7,frozenset,immutable,"Unordered set, contains no duplicates; can con...","frozenset([4.0, 'string', True])"
8,int,immutable,Integer of unlimited magnitude[114],42
9,list,mutable,"List, can contain mixed types","[4.0, 'string', True] []"


##### **Carga del Dataframe como un string** 

In [40]:
from pprint import pprint
data = df_tabla.to_string()
print(type(data))
pprint(data)

<class 'str'>
('                        Type '
 'Mutability                                                                                                                                                                      '
 'Description                                                                    '
 'Syntax examples\n'
 '0                       bool  '
 'immutable                                                                                                                                                                    '
 'Boolean '
 'value                                                                         '
 'True False\n'
 '1                  bytearray    '
 'mutable                                                                                                                                                                '
 'Sequence of bytes  bytearray(b\'Some ASCII\') bytearray(b"Some ASCII") '
 'bytearray([119, 105, 107, 105])\n'
 '2                      byte

##### **Test**

In [44]:
s4 = re.findall(r"\n\d{1,}\s{1,}(\w{7,})\s{1,}immutable", data)
s4

['complex', 'frozenset']

##### **Desarrollo**

In [43]:
# No es necesario especificar el \n como inicio de linea
s6 = re.findall(r"\d{1,}\s{1,}(\w{7,})\s{1,}immutable", data)
s6

['complex', 'frozenset']

#### **`Enunciado 7`: Queremos obtener la descripción de aquellos tipos de datos que solo tengan entre 1 y 3 caracteres y que sean inmutables**

##### **Carga de tabla HTML en un Dataframe** 

Nos muestra en una `lista` de **TODAS LAS TABLAS QUE SE ENCUENTRAN EN LA URL** en distintos Dataframes:

In [45]:
import pandas
import re

url = "https://en.wikipedia.org/wiki/Python_(programming_language)"
df = pandas.read_html(url)
df

[                                                    0  \
 0                                                 NaN   
 1                                            Paradigm   
 2                                         Designed by   
 3                                           Developer   
 4                                      First appeared   
 5                                                 NaN   
 6                                      Stable release   
 7                                   Typing discipline   
 8                                                  OS   
 9                                             License   
 10                                Filename extensions   
 11                                            Website   
 12                              Major implementations   
 13  CPython, PyPy, Stackless Python, MicroPython, ...   
 14                                           Dialects   
 15                       Cython, RPython, Starlark[9]   
 16           

Vamos a utilizar el segundo elemento de la lista que corresponde a la siguiente tabla:

In [46]:
df_tabla = pandas.read_html(url)[1]
df_tabla

,Type,Mutability,Description,Syntax examples
0,bool,immutable,Boolean value,True False
1,bytearray,mutable,Sequence of bytes,"bytearray(b'Some ASCII') bytearray(b""Some ASCI..."
2,bytes,immutable,Sequence of bytes,"b'Some ASCII' b""Some ASCII"" bytes([119, 105, 1..."
3,complex,immutable,Complex number with real and imaginary parts,3+2.7j 3 + 2.7j
4,dict,mutable,Associative array (or dictionary) of key and v...,"{'key1': 1.0, 3: False} {}"
5,types.EllipsisType,immutable,An ellipsis placeholder to be used as an index...,... Ellipsis
6,float,immutable,Double-precision floating-point number. The pr...,1.33333
7,frozenset,immutable,"Unordered set, contains no duplicates; can con...","frozenset([4.0, 'string', True])"
8,int,immutable,Integer of unlimited magnitude[114],42
9,list,mutable,"List, can contain mixed types","[4.0, 'string', True] []"


##### **Carga del Dataframe como un string** 

In [47]:
from pprint import pprint
data = df_tabla.to_string()
print(type(data))
pprint(data)

<class 'str'>
('                        Type '
 'Mutability                                                                                                                                                                      '
 'Description                                                                    '
 'Syntax examples\n'
 '0                       bool  '
 'immutable                                                                                                                                                                    '
 'Boolean '
 'value                                                                         '
 'True False\n'
 '1                  bytearray    '
 'mutable                                                                                                                                                                '
 'Sequence of bytes  bytearray(b\'Some ASCII\') bytearray(b"Some ASCII") '
 'bytearray([119, 105, 107, 105])\n'
 '2                      byte

##### **Test**

In [52]:
s7 = re.findall(r"\n\d{1,}\s{1,}\w{1,3}\s{1,}immutable\s{1,}(.+?)\s{2,}", data)
s7

['Integer of unlimited magnitude[114]',
 'A character string: sequence of Unicode codepoints']

##### **Desarrollo**

In [53]:
# No es necesario especificar el \n como inicio de linea
s7= re.findall(r"\d{1,}\s{1,}\w{1,3}\s{1,}immutable\s{1,}(.+?)\s{2,}", data)
s7

['Integer of unlimited magnitude[114]',
 'A character string: sequence of Unicode codepoints']

#### **`Enunciado 8`: Queremos obtener todos los tipos de datos donde su ID sea un número par incluyendo el 0**

##### **Carga de tabla HTML en un Dataframe** 

Nos muestra en una `lista` de **TODAS LAS TABLAS QUE SE ENCUENTRAN EN LA URL** en distintos Dataframes:

In [54]:
import pandas
import re

url = "https://en.wikipedia.org/wiki/Python_(programming_language)"
df = pandas.read_html(url)
df

[                                                    0  \
 0                                                 NaN   
 1                                            Paradigm   
 2                                         Designed by   
 3                                           Developer   
 4                                      First appeared   
 5                                                 NaN   
 6                                      Stable release   
 7                                   Typing discipline   
 8                                                  OS   
 9                                             License   
 10                                Filename extensions   
 11                                            Website   
 12                              Major implementations   
 13  CPython, PyPy, Stackless Python, MicroPython, ...   
 14                                           Dialects   
 15                       Cython, RPython, Starlark[9]   
 16           

Vamos a utilizar el segundo elemento de la lista que corresponde a la siguiente tabla:

In [55]:
df_tabla = pandas.read_html(url)[1]
df_tabla

,Type,Mutability,Description,Syntax examples
0,bool,immutable,Boolean value,True False
1,bytearray,mutable,Sequence of bytes,"bytearray(b'Some ASCII') bytearray(b""Some ASCI..."
2,bytes,immutable,Sequence of bytes,"b'Some ASCII' b""Some ASCII"" bytes([119, 105, 1..."
3,complex,immutable,Complex number with real and imaginary parts,3+2.7j 3 + 2.7j
4,dict,mutable,Associative array (or dictionary) of key and v...,"{'key1': 1.0, 3: False} {}"
5,types.EllipsisType,immutable,An ellipsis placeholder to be used as an index...,... Ellipsis
6,float,immutable,Double-precision floating-point number. The pr...,1.33333
7,frozenset,immutable,"Unordered set, contains no duplicates; can con...","frozenset([4.0, 'string', True])"
8,int,immutable,Integer of unlimited magnitude[114],42
9,list,mutable,"List, can contain mixed types","[4.0, 'string', True] []"


##### **Carga del Dataframe como un string** 

In [56]:
from pprint import pprint
data = df_tabla.to_string()
print(type(data))
pprint(data)

<class 'str'>
('                        Type '
 'Mutability                                                                                                                                                                      '
 'Description                                                                    '
 'Syntax examples\n'
 '0                       bool  '
 'immutable                                                                                                                                                                    '
 'Boolean '
 'value                                                                         '
 'True False\n'
 '1                  bytearray    '
 'mutable                                                                                                                                                                '
 'Sequence of bytes  bytearray(b\'Some ASCII\') bytearray(b"Some ASCII") '
 'bytearray([119, 105, 107, 105])\n'
 '2                      byte

##### **Test**

In [64]:
string = "0\n1\n2\n3\n4\n5\n6\n7\n8\n9\n10\n11\n12"
s = re.findall(r"\d*[02468]", string)
print(s)

['0', '2', '4', '6', '8', '10', '12']


In [60]:
s8 = re.findall(r"\n\d*[02468]\s{1,}(\w+?)\s{1,}.+", data)
s8

['bool', 'bytes', 'dict', 'float', 'int', 'range', 'str']

##### **Desarrollo**

In [61]:
# No es necesario especificar el \n como inicio de linea
s8 = re.findall(r"\d*[02468]\s{1,}(\w+?)\s{1,}.+", data)
s8

['bool', 'bytes', 'dict', 'float', 'int', 'range', 'str']

#### **`Enunciado 9`: Queremos obtener todos los tipos de datos donde su ID sea solo de 1 digito y que contenga un numero de 3 digitos ya sea en su descripcion o en Syntax examples**

##### **Carga de tabla HTML en un Dataframe** 

Nos muestra en una `lista` de **TODAS LAS TABLAS QUE SE ENCUENTRAN EN LA URL** en distintos Dataframes:

In [65]:
import pandas
import re

url = "https://en.wikipedia.org/wiki/Python_(programming_language)"
df = pandas.read_html(url)
df

[                                                    0  \
 0                                                 NaN   
 1                                            Paradigm   
 2                                         Designed by   
 3                                           Developer   
 4                                      First appeared   
 5                                                 NaN   
 6                                      Stable release   
 7                                   Typing discipline   
 8                                                  OS   
 9                                             License   
 10                                Filename extensions   
 11                                            Website   
 12                              Major implementations   
 13  CPython, PyPy, Stackless Python, MicroPython, ...   
 14                                           Dialects   
 15                       Cython, RPython, Starlark[9]   
 16           

Vamos a utilizar el segundo elemento de la lista que corresponde a la siguiente tabla:

In [66]:
df_tabla = pandas.read_html(url)[1]
df_tabla

,Type,Mutability,Description,Syntax examples
0,bool,immutable,Boolean value,True False
1,bytearray,mutable,Sequence of bytes,"bytearray(b'Some ASCII') bytearray(b""Some ASCI..."
2,bytes,immutable,Sequence of bytes,"b'Some ASCII' b""Some ASCII"" bytes([119, 105, 1..."
3,complex,immutable,Complex number with real and imaginary parts,3+2.7j 3 + 2.7j
4,dict,mutable,Associative array (or dictionary) of key and v...,"{'key1': 1.0, 3: False} {}"
5,types.EllipsisType,immutable,An ellipsis placeholder to be used as an index...,... Ellipsis
6,float,immutable,Double-precision floating-point number. The pr...,1.33333
7,frozenset,immutable,"Unordered set, contains no duplicates; can con...","frozenset([4.0, 'string', True])"
8,int,immutable,Integer of unlimited magnitude[114],42
9,list,mutable,"List, can contain mixed types","[4.0, 'string', True] []"


##### **Carga del Dataframe como un string** 

In [67]:
from pprint import pprint
data = df_tabla.to_string()
print(type(data))
pprint(data)

<class 'str'>
('                        Type '
 'Mutability                                                                                                                                                                      '
 'Description                                                                    '
 'Syntax examples\n'
 '0                       bool  '
 'immutable                                                                                                                                                                    '
 'Boolean '
 'value                                                                         '
 'True False\n'
 '1                  bytearray    '
 'mutable                                                                                                                                                                '
 'Sequence of bytes  bytearray(b\'Some ASCII\') bytearray(b"Some ASCII") '
 'bytearray([119, 105, 107, 105])\n'
 '2                      byte

##### **Test**

In [92]:
string = "0\n1\n2\n3\n4\n5\n6\n7\n8\n9\n10\n11\n12"
s = re.findall(r"\d", string)
print(s)

['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '1', '0', '1', '1', '1', '2']


In [75]:
string = "0\n1\n2\n3\n4\n5\n6\n7\n8\n9\n10\n11\n12"
s = re.findall(r"\d{1}", string)
print(s)

['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '1', '0', '1', '1', '1', '2']


In [91]:
string = "0\n1\n2\n3\n4\n5\n6\n7\n8\n9\n10\n11\n12"
s = re.findall(r"\d{2}", string)
print(s)

['10', '11', '12']


In [77]:
s9 = re.findall(r"\d{1}\s{1,}(\w+?)\s{1,}.*\d{3}.*", data)
s9

['bytearray', 'bytes', 'float', 'int', 'range']

##### **Desarrollo**

In [79]:
# Si es necesario especificar el \n como inicio de linea
s9 = re.findall(r"\n[0-9]\s{1,}(\w+?)\s{1,}.*\d{3}.*", data)
s9

['bytearray', 'bytes', 'float', 'int']

#### **`Enunciado 10`: Queremos obtener todos los tipos de datos que sean inmutables, si y solo si, no contengan un numero, cualquiera sea este, ya sea en su descripción o en Syntax examples**

##### **Carga de tabla HTML en un Dataframe** 

Nos muestra en una `lista` de **TODAS LAS TABLAS QUE SE ENCUENTRAN EN LA URL** en distintos Dataframes:

In [81]:
import pandas
import re

url = "https://en.wikipedia.org/wiki/Python_(programming_language)"
df = pandas.read_html(url)
df

[                                                    0  \
 0                                                 NaN   
 1                                            Paradigm   
 2                                         Designed by   
 3                                           Developer   
 4                                      First appeared   
 5                                                 NaN   
 6                                      Stable release   
 7                                   Typing discipline   
 8                                                  OS   
 9                                             License   
 10                                Filename extensions   
 11                                            Website   
 12                              Major implementations   
 13  CPython, PyPy, Stackless Python, MicroPython, ...   
 14                                           Dialects   
 15                       Cython, RPython, Starlark[9]   
 16           

Vamos a utilizar el segundo elemento de la lista que corresponde a la siguiente tabla:

In [82]:
df_tabla = pandas.read_html(url)[1]
df_tabla

,Type,Mutability,Description,Syntax examples
0,bool,immutable,Boolean value,True False
1,bytearray,mutable,Sequence of bytes,"bytearray(b'Some ASCII') bytearray(b""Some ASCI..."
2,bytes,immutable,Sequence of bytes,"b'Some ASCII' b""Some ASCII"" bytes([119, 105, 1..."
3,complex,immutable,Complex number with real and imaginary parts,3+2.7j 3 + 2.7j
4,dict,mutable,Associative array (or dictionary) of key and v...,"{'key1': 1.0, 3: False} {}"
5,types.EllipsisType,immutable,An ellipsis placeholder to be used as an index...,... Ellipsis
6,float,immutable,Double-precision floating-point number. The pr...,1.33333
7,frozenset,immutable,"Unordered set, contains no duplicates; can con...","frozenset([4.0, 'string', True])"
8,int,immutable,Integer of unlimited magnitude[114],42
9,list,mutable,"List, can contain mixed types","[4.0, 'string', True] []"


##### **Carga del Dataframe como un string** 

In [83]:
from pprint import pprint
data = df_tabla.to_string()
print(type(data))
pprint(data)

<class 'str'>
('                        Type '
 'Mutability                                                                                                                                                                      '
 'Description                                                                    '
 'Syntax examples\n'
 '0                       bool  '
 'immutable                                                                                                                                                                    '
 'Boolean '
 'value                                                                         '
 'True False\n'
 '1                  bytearray    '
 'mutable                                                                                                                                                                '
 'Sequence of bytes  bytearray(b\'Some ASCII\') bytearray(b"Some ASCII") '
 'bytearray([119, 105, 107, 105])\n'
 '2                      byte

##### **Test**

In [89]:
string = "0\n1\n2\n3\n4\n5\n6\n7\n8\n9\n10\n11\n12"
s = re.findall(r"\d{1,}", string)
print(s)

['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12']


In [88]:
s10 = re.findall(r"\n\d{1,}\s{1,}(\w+?)\s{1,}immutable\s{1,}(?!.*\d{1,}.*)", data)
s10

['bool', 'str']

##### **Desarrollo**

In [87]:
# No es necesario especificar el \n como inicio de linea
s10 = re.findall(r"\d{1,}\s{1,}(\w+?)\s{1,}immutable\s{1,}(?!.*\d{1,}.*)", data)
s10

['bool', 'str']